In [0]:
CREATE OR REFRESH STREAMING TABLE first_data_engineering_project.silver.silver_orders
AS
WITH filtered_bronze AS (
    SELECT 
        order_id,
        customer_id,
        store_id,
        order_date,
        promotion_id,
        ingestion_timestamp,
        source_file
    FROM STREAM(first_data_engineering_project.bronze.bronze_orders)
    WHERE 
        -- Remove NULL values in critical columns
        order_id IS NOT NULL
        AND customer_id IS NOT NULL
        AND store_id IS NOT NULL
        AND order_date IS NOT NULL
        -- Validate data quality - ensure positive IDs
        AND order_id > 0
        AND customer_id > 0
        AND store_id > 0
),
deduped_orders AS (
    SELECT 
        *,
        -- Remove duplicate orders (keep the most recent by ingestion_timestamp)
        ROW_NUMBER() OVER (
            PARTITION BY order_id 
            ORDER BY ingestion_timestamp DESC
        ) AS row_num
    FROM filtered_bronze
)
SELECT 
    order_id,
    customer_id,
    store_id,
    order_date,
    promotion_id,
    ingestion_timestamp,
    source_file,
    -- Add silver layer metadata
    CURRENT_TIMESTAMP() AS silver_processed_timestamp,
    'VALID' AS data_quality_flag
FROM deduped_orders
WHERE row_num = 1;